In [47]:
from pyspark.sql import (
    functions as f,
    SparkSession,
    types as t
)

spark = SparkSession.builder.appName("df_missing_data").getOrCreate()
df = spark.read.csv(
    "file:///home/jovyan/work/sample/null_data.csv", header=True, inferSchema=True)
# df.show()

# DataFrame.na: Returns a DataFrameNaFunctions for handling missing values.
# DataFrame.dropna(how='any', thresh=None, subset=None)[source]: Returns a new DataFrame omitting rows with null values. DataFrame.dropna() and DataFrameNaFunctions.drop() are aliases of each other.
#   how: 'any’ or ‘all’. If ‘any’, drop a row if it contains any nulls. If ‘all’, drop a row only if all its values are null.
#   thresh: default None If specified, drop rows that have less than thresh non-null values. This overwrites the how parameter.
#   subset: optional list of column names to consider.

# PySpark에서 null을 핸들링 하기 위해서 제공해주는 함수가 na.drop
# 해당 함수에 3가지 파라미터를 넣어줄 수 있는데
# 1. how = any일 경우 어떤 value값이라도 null값을 가지고 있으면 drop을 하라는것, all은 모든 데이터가 빈칸일 경우에만 Drop 하라는 뜻
# 2. thresh = row에 null이 몇개있는지에 따라서 drop하게 하는 파라미터
# 3. subset = 컬럼을 지정해서 해당 컬럼에서 빈칸이 있을 경우에는 drop

# df.na.drop(how="any").show()
# df.na.drop(thresh=2).show()
# df.na.drop(subset=["salary"]).show()

df.printSchema()

# # fill string
# 만약 비어있는 데이터가 있으면 채워서 사용하고 싶을때 fill을 사용
# 스파크는 컬럼의 필드 타입을 다 알고있는데 string 값을 넣게되면 string 컬럼이 비어있는곳에 전부 채워주고 integer값을 넣게되면 비어있는 integer값을 채워줌
# df.na.fill("engineer").show()

# # fill integer
# df.na.fill(0).show()

# # fill the subset
# subset을 지정해서 해당 컬럼의 값에 null이 있으면 채워줄 수도 있음
# df.na.fill("NA", subset=["occupation"]).show()

# # fill the mean value
# # mean function은 array내부의 특정 column의 평균값을 구하는 함수 왜 mean인지 몰겟슴
# mean_value = df.select(f.mean(df['salary'])).collect()

# # print(mean_value[0][0])

# df.na.fill(mean_value[0][0], subset=["salary"]).show()



# # Date parsing
spark = SparkSession.builder.appName("df_manage_date").getOrCreate()
df = spark.read.csv(
    "file:///home/jovyan/work/sample/date_parsing.csv", header=True, inferSchema=True)
# df.show()
# # show year
# # year 값 보여주기
# df.select(f.year('date')).show()

# # show month
# df.select(f.month('date')).show()

# # show day
# Day는 dayofmonth를 사용
# dayofyear은 오늘이 올해의 몇번째 날인지 보여줌
# df.select(f.dayofmonth('date').alias('day')).show()
# df.select(f.dayofyear('date').alias('day')).show()

# 각 년도의 number 평균 구하기
df = df.withColumn("year", f.year('date')).groupBy("year").mean("number").withColumnRenamed("avg(number)", "avg")
# df.show()
df.select("year", f.format_number("avg", 2).alias("avg")).show()

root
 |-- id: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- salary: integer (nullable = true)

+----+--------+
|year|     avg|
+----+--------+
|2022|2,540.67|
|2021|2,195.68|
+----+--------+

